## In this notebook, we partition the _Corpus Nummorum_ data and train/validate/test our computer-vision model

Plan of attack
* Importing some libraries
* Creating train/validate/test split
* Training and validation with ImageNet 21k model
* Fine-tuning
* Testing
* Saving model checkpoint as .pth file

In [4]:
# from google.colab import runtime
# runtime.unassign()

In [1]:
!pip install import-ipynb
!pip install iterative-stratification -q
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 12.4 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append('/content/drive/MyDrive/coins_project/')
from utils import *

Mounted at /content/drive


Objects from `data_exploration.ipynb`

In [4]:
sample_motifs = ["eagle", "throne", "snake", "bull", "horse", "star", "head"]

image_path = Path('/content/drive/MyDrive/coins_project/CoinsDataset/CN_dataset_nlp_objects')
motif_root = Path(image_path)
image_exts = {".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff"}

rows = []

for motif_dir in motif_root.iterdir():
    if not motif_dir.is_dir():
        continue

    motif = motif_dir.name

    for img_path in motif_dir.rglob("*"):
        if img_path.suffix.lower() in image_exts:
            parsed = parse_coin_filename(img_path)

            rows.append({
                **parsed,
                "motif": motif,
                "image_path_in_motif_folder": str(img_path),
            })

long_df = pd.DataFrame(rows)

id_cols = ["image_id", "filename", "coin_id", "type_id", "side"]

base_df = (
    long_df
    .groupby(id_cols)["image_path_in_motif_folder"]
    .first()
    .reset_index()
    .rename(columns={"image_path_in_motif_folder": "image_path"})
)

label_df = (
    long_df[long_df["motif"].isin(sample_motifs)]
    .assign(value=1)
    .pivot_table(
        index=id_cols,
        columns="motif",
        values="value",
        aggfunc="max",
        fill_value=0,
    )
    .reset_index()
)

label_df.columns.name = None

df = base_df.merge(label_df, on=id_cols, how="left")
df[sample_motifs] = df[sample_motifs].fillna(0).astype("float32")

Partition into training, validation, and testing data according to 64/16/20 split

In [5]:
group_df = (
    df.groupby("coin_id")[sample_motifs]
    .max()
    .reset_index()
)

X = group_df["coin_id"].values
Y = group_df[sample_motifs].values

splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

trainval_idx, test_idx = next(splitter.split(X, Y))

trainval_groups = group_df.iloc[trainval_idx].copy()
test_groups = group_df.iloc[test_idx].copy()

splitter2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

X_tv = trainval_groups["coin_id"].values
Y_tv = trainval_groups[sample_motifs].values

train_idx, val_idx = next(splitter2.split(X_tv, Y_tv))

train_groups = set(trainval_groups.iloc[train_idx]["coin_id"])
val_groups = set(trainval_groups.iloc[val_idx]["coin_id"])
test_groups = set(test_groups["coin_id"])

df["split"] = "none"
df.loc[df["coin_id"].isin(train_groups), "split"] = "train"
df.loc[df["coin_id"].isin(val_groups), "split"] = "val"
df.loc[df["coin_id"].isin(test_groups), "split"] = "test"

df["split"].value_counts()

,count
split,
train,34906
test,10898
val,8685


In [6]:
df.groupby("split")[sample_motifs].sum()

,eagle,throne,snake,bull,horse,star,head
split,,,,,,,
test,317.0,281.0,479.0,265.0,493.0,122.0,3612.0
train,999.0,901.0,1538.0,861.0,1565.0,381.0,11472.0
val,248.0,222.0,384.0,216.0,387.0,94.0,2894.0


In [8]:
# import timm
# from timm.data import resolve_model_data_config, create_transform

# sample_motifs.append("head")
# # sample_motifs.append("spear")

# num_motifs = len(sample_motifs)

# model = timm.create_model(
#     "vit_base_patch16_224.augreg_in21k",
#     pretrained=True,
#     num_classes=num_motifs
# )

# data_config = resolve_model_data_config(model)

# train_tfms = create_transform(**data_config, is_training=True)
# eval_tfms = create_transform(**data_config, is_training=False)

In [10]:
# # from pathlib import Path

# src = Path(image_path)  # your Drive folder containing motif/image folders
# tar_path = Path("/content/drive/MyDrive/coin_images.tar")

# print("source:", src)
# print("tar:", tar_path)

# !tar -czf "$tar_path" -C "$src" .

In [11]:
# local_root = Path("/content/coin_images")
# local_root.mkdir(parents=True, exist_ok=True)

In [12]:
# !tar -xf "$tar_path" -C "$local_root"

In [13]:
# !mkdir -p /content/coin_images
# !tar -xf /content/drive/MyDrive/coin_images.tar -C /content/coin_images

In [14]:
# old_root = str(Path(image_path))
# new_root = str(local_root)

# df["image_path"] = df["image_path"].str.replace(
#     old_root,
#     new_root,
#     regex=False
# )

# long_df["image_path_in_motif_folder"] = long_df["image_path_in_motif_folder"].str.replace(
#     old_root,
#     new_root,
#     regex=False
# )

In [15]:
df["image_exists"] = df["image_path"].apply(lambda p: Path(p).exists())
df["image_exists"].value_counts()

,count
image_exists,
True,54489


In [16]:
df

,image_id,filename,coin_id,type_id,side,image_path,bull,eagle,head,horse,snake,star,throne,split,image_exists
0,10001_obv,CN_type_10733_MK_18234616_cn_coin_10001_o_obv.jpg,10001,10733,obv,/content/drive/MyDrive/coins_project/CoinsData...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,train,True
1,10001_rev,CN_type_10733_MK_18234616_cn_coin_10001_o_rev.jpg,10001,10733,rev,/content/drive/MyDrive/coins_project/CoinsData...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,train,True
2,10009_obv,CN_type_8658_MK_18235374_cn_coin_10009_o_obv.jpg,10009,8658,obv,/content/drive/MyDrive/coins_project/CoinsData...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,train,True
3,10009_rev,CN_type_8658_MK_18235374_cn_coin_10009_o_rev.jpg,10009,8658,rev,/content/drive/MyDrive/coins_project/CoinsData...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,train,True
4,1000_obv,CN_type_8499_cn_coin_1000_p_obv.jpg,1000,8499,obv,/content/drive/MyDrive/coins_project/CoinsData...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,test,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54484,9993_rev,CN_type_8281_MK_18208282_cn_coin_9993_o_rev.jpg,9993,8281,rev,/content/drive/MyDrive/coins_project/CoinsData...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,val,True
54485,999_obv,CN_type_8499_cn_coin_999_p_obv.jpg,999,8499,obv,/content/drive/MyDrive/coins_project/CoinsData...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,val,True
54486,999_rev,CN_type_8499_cn_coin_999_p_rev.jpg,999,8499,rev,/content/drive/MyDrive/coins_project/CoinsData...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,val,True
54487,99_obv,CN_type_3987_cn_coin_99_p_obv.jpg,99,3987,obv,/content/drive/MyDrive/coins_project/CoinsData...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,test,True


In [17]:
import timm
from timm.data import resolve_model_data_config, create_transform

num_motifs = len(sample_motifs)

model = timm.create_model(
    "vit_base_patch16_224.augreg_in21k",
    pretrained=True,
    num_classes=num_motifs
)

data_config = resolve_model_data_config(model)

train_tfms = create_transform(**data_config, is_training=True)
eval_tfms = create_transform(**data_config, is_training=False)

model.safetensors: reconstructing file:   0%|          |  0.00B /  410MB            

model.safetensors: downloading bytes:           |  0.00B            

In [18]:
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

train_ds = CoinMotifDataset(train_df, sample_motifs, train_tfms)
val_ds = CoinMotifDataset(val_df, sample_motifs, eval_tfms)
test_ds = CoinMotifDataset(test_df, sample_motifs, eval_tfms)

In [19]:
test_df

,image_id,filename,coin_id,type_id,side,image_path,bull,eagle,head,horse,snake,star,throne,split,image_exists
4,1000_obv,CN_type_8499_cn_coin_1000_p_obv.jpg,1000,8499,obv,/content/drive/MyDrive/coins_project/CoinsData...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,test,True
5,1000_rev,CN_type_8499_cn_coin_1000_p_rev.jpg,1000,8499,rev,/content/drive/MyDrive/coins_project/CoinsData...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,test,True
20,10027_obv,CN_type_11182_MK_18241034_cn_coin_10027_o_obv.jpg,10027,11182,obv,/content/drive/MyDrive/coins_project/CoinsData...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,test,True
21,10027_obv,CN_type_11182_cn_coin_10027_p_obv.jpg,10027,11182,obv,/content/drive/MyDrive/coins_project/CoinsData...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,test,True
22,10027_rev,CN_type_11182_MK_18241034_cn_coin_10027_o_rev.jpg,10027,11182,rev,/content/drive/MyDrive/coins_project/CoinsData...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,test,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54464,9981_rev,CN_type_2098_cn_coin_9981_p_rev.jpg,9981,2098,rev,/content/drive/MyDrive/coins_project/CoinsData...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,test,True
54477,9988_obv,CN_type_2105_cn_coin_9988_p_obv.jpg,9988,2105,obv,/content/drive/MyDrive/coins_project/CoinsData...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,test,True
54478,9988_rev,CN_type_2105_cn_coin_9988_p_rev.jpg,9988,2105,rev,/content/drive/MyDrive/coins_project/CoinsData...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,test,True
54487,99_obv,CN_type_3987_cn_coin_99_p_obv.jpg,99,3987,obv,/content/drive/MyDrive/coins_project/CoinsData...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,test,True


Preparing and training the model

In [20]:
batch_size = 32

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

In [22]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

train_labels = torch.tensor(train_df[sample_motifs].values.astype("float32"))

pos = train_labels.sum(dim=0)
neg = len(train_labels) - pos
pos_weight = (neg / pos.clamp(min=1)).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [23]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

In [ ]:
# Phase 1: train only the new classifier head
for p in model.parameters():
    p.requires_grad = False

for p in model.get_classifier().parameters():
    p.requires_grad = True

count_trainable(model)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4,
)

best_score = -np.inf
best_state = None

best_score, best_state = run_training_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    num_epochs=5,
    phase="Head training",
    best_score=best_score,
    best_state=best_state,
)

Fine-tuning

In [ ]:
# Phase 2: fine-tune last 4 ViT blocks + final norm + classifier head
for p in model.parameters():
    p.requires_grad = False

for p in model.blocks[-4:].parameters():
    p.requires_grad = True

for p in model.norm.parameters():
    p.requires_grad = True

for p in model.get_classifier().parameters():
    p.requires_grad = True

count_trainable(model)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr= 1e-5,
    weight_decay=1e-4,
)

best_score, best_state = run_training_phase(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    num_epochs=10,
    phase="Fine-tuning",
    best_score=best_score,
    best_state=best_state,
)

model.load_state_dict(best_state)
print(f"Best validation macro AP: {best_score:.4f}")

In [ ]:
test_probs, test_y = predict(
    model=model,
    loader=test_loader,
    device=device,
    desc="Test",
)

for j, motif in enumerate(sample_motifs):
    positives = int(test_y[:, j].sum())

    if positives > 0:
        ap = average_precision_score(test_y[:, j], test_probs[:, j])
        print(f"{motif}: AP={ap:.4f}, positives={positives}")
    else:
        print(f"{motif}: AP=undefined, positives=0")

Now we survey our handiwork and verify that our model is picking up on the right things

In [ ]:
show_top_predictions(
    model=model,
    dataset=test_ds,
    motif_cols=sample_motifs,
    device=device,
    n=20,
    top_k=3,
)

Saves the model checkpoint as a .pth file

In [ ]:
save_path = Path("/content/drive/MyDrive/coin_motif_vit_checkpoint_new.pth")

checkpoint = {
    "model_name": "vit_base_patch16_224.augreg_in21k",
    "model_state_dict": model.state_dict(),
    "sample_motifs": sample_motifs,
    "thresholds": thresholds if "thresholds" in globals() else None,
    "best_score": best_score if "best_score" in globals() else None,
}

torch.save(checkpoint, save_path)

print("saved to:", save_path)

In [ ]:
# import torch
# import timm

# checkpoint = torch.load(
#     "/content/drive/MyDrive/coin_motif_vit_checkpoint.pth",
#     map_location="cuda" if torch.cuda.is_available() else "cpu"
# )

# sample_motifs = checkpoint["sample_motifs"]
# num_motifs = len(sample_motifs)

# model = timm.create_model(
#     checkpoint["model_name"],
#     pretrained=False,
#     num_classes=num_motifs
# )

# model.load_state_dict(checkpoint["model_state_dict"])

# device = "cuda" if torch.cuda.is_available() else "cpu"
# model = model.to(device)
# model.eval()

# thresholds = checkpoint.get("thresholds")

In [ ]:
# with torch.no_grad():
#     logits = model(imgs.to(device))
#     probs = torch.sigmoid(logits)